# 02 Initialization and depth: vanishing and exploding signals

Lecture 10, training and generalization. Book chapter 4 (initialization and
the flow of signals).

**Goals**

- Measure the scale of the pre-activations through a deep MLP with
  `glassnn.analysis.activation_stats`.
- Compare too-small, Xavier, Kaiming (He) and too-large initializations, for
  ReLU and tanh networks.

**Expected finding** (asserted in `tests/test_examples.py::test_02_*`): in a
20-layer ReLU network of width 256, the standard deviation of the
pre-activations is multiplied at every layer by $\sqrt{n\,\mathrm{Var}(w)/2}$.
Only Kaiming initialization ($\mathrm{Var}(w) = 2/n$) keeps it constant; the
others make it vanish or explode geometrically.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from glassnn import manual_seed, nn
from glassnn.analysis import activation_stats
from glassnn.nn import init

WIDTH, DEPTH = 256, 20
X = np.random.default_rng(0).normal(size=(512, WIDTH))

INITIALIZERS = {
    "too small, N(0, 0.01²)": lambda w: init.normal_(w, std=0.01),
    "default of Linear": lambda w: init.kaiming_uniform_(w, a=np.sqrt(5)),
    "Xavier normal": init.xavier_normal_,
    "Kaiming normal (relu)": lambda w: init.kaiming_normal_(w, nonlinearity="relu"),
    "too large, N(0, 0.1²)": lambda w: init.normal_(w, std=0.1),
}


def deep_mlp(activation, initializer):
    # DEPTH linear layers of width WIDTH, each followed by the activation.
    manual_seed(0)
    layers = []
    for _ in range(DEPTH):
        layer = nn.Linear(WIDTH, WIDTH)
        initializer(layer.weight)
        init.zeros_(layer.bias)
        layers += [layer, activation()]
    return nn.Sequential(*layers)

`activation_stats(model, X)` runs the model without recording a graph and
returns, for every `Linear` layer (keyed by its name in the `Sequential`),
the mean and the standard deviation of its output $Z_l$.

In [ ]:
stats = activation_stats(deep_mlp(nn.ReLU, init.kaiming_normal_), X)
list(stats.items())[:3]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
ratios = {}
for ax, activation in zip(axes, [nn.ReLU, nn.Tanh], strict=True):
    for label, initializer in INITIALIZERS.items():
        stats = activation_stats(deep_mlp(activation, initializer), X)
        stds = [s.std for s in stats.values()]
        ratios[(activation.__name__, label)] = stds[-1] / stds[0]
        ax.semilogy(range(1, DEPTH + 1), stds, marker=".", label=label)
    ax.set_title(activation.__name__)
    ax.set_xlabel("layer $l$")
axes[0].set_ylabel("std of $Z_l$")
axes[1].legend(fontsize=8)
plt.show()

In [ ]:
for (activation, label), ratio in ratios.items():
    print(f"{activation:5s} {label:25s} std(Z_20) / std(Z_1) = {ratio:.2e}")

**Finding.** With ReLU, the ratio between the last and the first layer is
close to 1 only for Kaiming initialization. Xavier loses a factor
$1/\sqrt 2$ per layer (it is designed for activations with slope 1 at 0),
PyTorch's default (variance $1/(3n)$) a factor $1/\sqrt 6$, and the two
fixed standard deviations a factor $0.11$ and $1.13$. With tanh, signals
cannot explode (tanh is bounded) and Xavier keeps a slowly decaying scale.

The backward pass has the same factor per layer (chapter 4), so the
gradients of the first layers vanish or explode in the same way.

**Exercise.** Train each ReLU network for a few hundred steps on a simple
regression problem with SGD. Which initializations train at all?